# 02. Feature Engineering

**Objective:** Construct features for the TCTR framework: Structured, Temporal Dynamics, Semantic Graph, and Contextual Exposure.

**Inputs:** Train, Validation, and Test datasets (from `01_data_preprocessing_and_temporal_split.parquet`).

**Outputs:** Feature-rich DataFrames, sequence tensors, and Graph Adjacency matrices.

In [1]:
import pandas as pd
import numpy as np
import os
import torch
from sentence_transformers import SentenceTransformer
import networkx as nx
from sklearn.neighbors import NearestNeighbors

# Set up paths relative to the notebooks directory inside Antigravity IDE
DATA_DIR = os.path.join("..", "Data", "TCTR_splits")
OUTPUT_DIR = os.path.join("..", "Data", "TCTR_features")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data directory: {os.path.abspath(DATA_DIR)}")
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

# Determine device for embedding generation
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device for embeddings: {device}")

Data directory: d:\NetShield\NetShieldAI\Data\TCTR_splits
Output directory: d:\NetShield\NetShieldAI\Data\TCTR_features
Using device for embeddings: cuda


## 1. Load Split Data

Load the datasets created in the previous notebook.

In [2]:
# Load the datasets created in Notebook 01
train_df = pd.read_parquet(os.path.join(DATA_DIR, "train_cve.parquet"))
val_df = pd.read_parquet(os.path.join(DATA_DIR, "val_cve.parquet"))
test_df = pd.read_parquet(os.path.join(DATA_DIR, "test_cve.parquet"))

print(f"Loaded datasets: Train({len(train_df)}), Val({len(val_df)}), Test({len(test_df)})")

Loaded datasets: Train(90566), Val(40484), Test(6976)


In [3]:
def compute_structured_features(df):
    """Extracts basic structured counts and lengths from the raw lists."""
    df = df.copy()
    
    # Text length feature
    df['desc_length'] = df['description'].fillna('').apply(len)
    
    # Complexity/Exposure proxies based on lists (handling empty lists safely)
    df['num_keywords'] = df['keywords'].apply(lambda x: len(x) if isinstance(x, np.ndarray) or isinstance(x, list) else 0)
    df['num_platforms'] = df['platforms'].apply(lambda x: len(x) if isinstance(x, np.ndarray) or isinstance(x, list) else 0)
    df['num_affected_products'] = df['affected_products'].apply(lambda x: len(x) if isinstance(x, np.ndarray) or isinstance(x, list) else 0)
    
    return df

print("Computing structured features...")
train_df = compute_structured_features(train_df)
val_df = compute_structured_features(val_df)
test_df = compute_structured_features(test_df)

Computing structured features...


## 2. Temporal Dynamics (EPSS Features)

Compute EPSS derivatives: velocity, acceleration, and momentum. 
*Note: In a real scenario, this requires longitudinal EPSS data. For this prototype, we'll derive them from current EPSS and time since publication.*

In [4]:
def compute_temporal_features(df, split_end_date):
    """
    Computes temporal features without future leakage by evaluating age 
    relative to the end of the split period, NOT 'today'.
    """
    df = df.copy()
    
    # Ensure dates are datetime objects
    df['published_date'] = pd.to_datetime(df['published_date'], utc=True)
    df['last_modified_date'] = pd.to_datetime(df['last_modified_date'], utc=True)
    horizon_date = pd.to_datetime(split_end_date, utc=True)
    
    # Age of the CVE at the time of the split horizon
    df['days_since_pub_at_horizon'] = (horizon_date - df['published_date']).dt.days.clip(lower=1)
    
    # How quickly was it modified after publication? (Indicator of active exploitation/patching)
    df['days_to_last_modify'] = (df['last_modified_date'] - df['published_date']).dt.days.clip(lower=0)
    
    # Mock EPSS / Base Score dynamics (since we don't have historical EPSS logs in this demo)
    # In production, replace this with actual historical EPSS joins
    score_col = 'epss' if 'epss' in df.columns else 'days_to_last_modify' # Fallback feature
    
    # Velocity proxies
    df['mock_threat_velocity'] = df[score_col] / df['days_since_pub_at_horizon']
    df['mock_threat_acceleration'] = df['mock_threat_velocity'] / df['days_since_pub_at_horizon']
    
    return df

print("Computing temporal dynamics...")
# Using the boundaries we defined in Notebook 1
train_df = compute_temporal_features(train_df, "2023-12-31")
val_df = compute_temporal_features(val_df, "2024-12-31")
test_df = compute_temporal_features(test_df, "2026-12-31")

Computing temporal dynamics...


## 3. Contextual Exposure (Text Embeddings)

Generate text embeddings for CVE descriptions using `all-MiniLM-L6-v2`.

In [5]:
# Initialize the model using the detected device (GPU/CPU)
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

def get_embeddings(df):
    descriptions = df['description'].fillna('').tolist()
    # Batch processing is handled automatically by encode, but we show the progress bar
    embeddings = model.encode(descriptions, show_progress_bar=True, batch_size=128)
    return embeddings

print("Generating embeddings for Train...")
train_embeddings = get_embeddings(train_df)
print("Generating embeddings for Val...")
val_embeddings = get_embeddings(val_df)
print("Generating embeddings for Test...")
test_embeddings = get_embeddings(test_df)

print(f"\nEmbeddings shape - Train: {train_embeddings.shape}")

Generating embeddings for Train...


Batches:   0%|          | 0/708 [00:00<?, ?it/s]

Generating embeddings for Val...


Batches:   0%|          | 0/317 [00:00<?, ?it/s]

Generating embeddings for Test...


Batches:   0%|          | 0/55 [00:00<?, ?it/s]


Embeddings shape - Train: (90566, 384)


## 4. Semantic Graph Features

Establish semantic similarity edges and calculate graph centrality.

In [6]:
def compute_graph_features(df, embeddings, n_neighbors=5):
    """
    Builds a sparse K-Nearest Neighbors graph using cosine similarity 
    and calculates graph metrics like Degree Centrality.
    """
    print(f"Building semantic graph for {len(df)} nodes...")
    df = df.copy()
    
    if len(df) < n_neighbors:
        df['semantic_centrality'] = 0.0
        return df
        
    # Fit KNN on the embeddings using cosine distance
    nbrs = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine', algorithm='brute').fit(embeddings)
    distances, indices = nbrs.kneighbors(embeddings)
    
    # Build NetworkX graph
    G = nx.Graph()
    G.add_nodes_from(range(len(df)))
    
    # Add edges for nearest neighbors (skip the first one as it's the node itself)
    edges = []
    for i in range(len(df)):
        for j in range(1, n_neighbors): 
            # Convert cosine distance to similarity weight
            weight = 1.0 - distances[i][j]
            if weight > 0.7:  # Only connect if highly similar
                edges.append((i, indices[i][j], weight))
                
    G.add_weighted_edges_from(edges)
    print(f"Graph constructed with {G.number_of_edges()} edges.")
    
    # Calculate Degree Centrality (how connected is this CVE to other similar vulnerabilities?)
    # Highly central nodes might indicate a systemic vulnerability class
    centrality = nx.degree_centrality(G)
    
    # Map back to dataframe
    df['semantic_centrality'] = df.index.map(centrality)
    df['semantic_centrality'] = df['semantic_centrality'].fillna(0)
    
    return df

train_df = compute_graph_features(train_df, train_embeddings)
val_df = compute_graph_features(val_df, val_embeddings)
test_df = compute_graph_features(test_df, test_embeddings)

Building semantic graph for 90566 nodes...
Graph constructed with 189323 edges.
Building semantic graph for 40484 nodes...
Graph constructed with 80422 edges.
Building semantic graph for 6976 nodes...
Graph constructed with 11159 edges.


## 5. Save Feature Data

Save the enriched DataFrames and embeddings.

In [7]:
# Save the enriched DataFrames
train_df.to_parquet(os.path.join(OUTPUT_DIR, "train_features.parquet"), index=False)
val_df.to_parquet(os.path.join(OUTPUT_DIR, "val_features.parquet"), index=False)
test_df.to_parquet(os.path.join(OUTPUT_DIR, "test_features.parquet"), index=False)

# Save the numpy embedding matrices (essential for Notebook 4: Advanced Neural Models)
np.save(os.path.join(OUTPUT_DIR, "train_embeddings.npy"), train_embeddings)
np.save(os.path.join(OUTPUT_DIR, "val_embeddings.npy"), val_embeddings)
np.save(os.path.join(OUTPUT_DIR, "test_embeddings.npy"), test_embeddings)

print("Features and embeddings successfully saved to D:\\NetShield\\NetShieldAI\\Data\\TCTR_features\\")

Features and embeddings successfully saved to D:\NetShield\NetShieldAI\Data\TCTR_features\
